<a href="https://colab.research.google.com/github/nicolaspromano/Trabalho-de-Conclusao-de-Curso-1/blob/main/Predicao_SFCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

caminho_tsv = '/content/drive/MyDrive/Colab_Notebooks/dataset/val_labels/participants.tsv'
caminho_imagens = '/content/drive/MyDrive/Colab_Notebooks/dataset/val_vbm'

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ARQUITETURA
class CNN3D_OpenBHB(nn.Module):
    def __init__(self, n_classes=1):
        super(CNN3D_OpenBHB, self).__init__()
        self.conv1 = nn.Conv3d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv3d(128, 256, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv3d(256, 256, kernel_size=3, stride=1, padding=1)
        self.conv6 = nn.Conv3d(256, 64, kernel_size=1, stride=1)

        self.batchnorm1 = nn.BatchNorm3d(32)
        self.batchnorm2 = nn.BatchNorm3d(64)
        self.batchnorm3 = nn.BatchNorm3d(128)
        self.batchnorm4 = nn.BatchNorm3d(256)
        self.batchnorm5 = nn.BatchNorm3d(256)
        self.batchnorm6 = nn.BatchNorm3d(64)

        self.maxpool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.avgpool = nn.AvgPool3d(kernel_size=(3, 4, 3), stride=1)
        self.dropout = nn.Dropout3d(p=0.5)
        self.relu = nn.ReLU()
        self.classifier = nn.Conv3d(64, n_classes, kernel_size=1, stride=1)

    def forward(self, x):
        x = self.relu(self.maxpool(self.batchnorm1(self.conv1(x))))
        x = self.relu(self.maxpool(self.batchnorm2(self.conv2(x))))
        x = self.relu(self.maxpool(self.batchnorm3(self.conv3(x))))
        x = self.relu(self.maxpool(self.batchnorm4(self.conv4(x))))
        x = self.relu(self.maxpool(self.batchnorm5(self.conv5(x))))

        x = self.relu(self.batchnorm6(self.conv6(x)))
        x = self.avgpool(x)
        x = self.dropout(x)
        x = self.classifier(x)
        x = x.view(x.shape[0], -1)
        return x


# DADOS
class OpenBHBDataset(Dataset):
    def __init__(self, tsv_file, img_dir):
        df_completo = pd.read_csv(tsv_file, sep='\t')
        self.img_dir = img_dir

        print("Filtrando pacientes com o nome real dos arquivos...")

        pacientes_validos = []
        for index, row in df_completo.iterrows():
            patient_id = row['participant_id']

            img_name = f"sub-{patient_id}_preproc-cat12vbm_desc-gm_T1w.npy"

            img_path = os.path.join(self.img_dir, img_name)

            if os.path.exists(img_path):
                pacientes_validos.append(row)

        self.labels_df = pd.DataFrame(pacientes_validos)

        print(f"Filtro concluído! Encontradas {len(self.labels_df)} imagens na pasta de um total de {len(df_completo)}.")

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        patient_id = self.labels_df.iloc[idx]['participant_id']
        age = self.labels_df.iloc[idx]['age']
        y_label = torch.tensor([age], dtype=torch.float32)

        img_name = f"sub-{patient_id}_preproc-cat12vbm_desc-gm_T1w.npy"

        img_path = os.path.join(self.img_dir, img_name)

        image_np = np.load(img_path)
        image_np = np.squeeze(image_np)

        image_tensor = torch.from_numpy(image_np).float()
        image_tensor = image_tensor.unsqueeze(0)

        return image_tensor, y_label


# TREINAMENTO
def treinar_modelo():
    # verificar se tem GPU disponível
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Treinando na unidade: {device}")

    dataset = OpenBHBDataset(tsv_file=caminho_tsv, img_dir=caminho_imagens)

    # lotes 4 pacientes por vez
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

    # instancia o modelo na placa de video
    modelo = CNN3D_OpenBHB().to(device)

    # correção
    criterio = nn.L1Loss()
    # otimizador
    otimizador = optim.Adam(modelo.parameters(), lr=0.0001)

    num_epocas = 15

    print("Iniciando o Treinamento...")
    for epoca in range(num_epocas):
        modelo.train()
        erro_total_epoca = 0.0

        # barra de progresso visual
        loop = tqdm(dataloader, total=len(dataloader), leave=True)

        for imagens, idades_reais in loop:
            imagens = imagens.to(device)
            idades_reais = idades_reais.to(device)

            otimizador.zero_grad()

            # passa as imagens pela rede
            idades_previstas = modelo(imagens)

            # calcula o erro
            erro = criterio(idades_previstas, idades_reais)

            # Backward propagation
            erro.backward()
            otimizador.step()

            erro_total_epoca += erro.item()

            loop.set_description(f"Época [{epoca+1}/{num_epocas}]")
            loop.set_postfix(Erro_Médio_Anos=erro.item())

        erro_medio = erro_total_epoca / len(dataloader)
        print(f"\nFim da Época {epoca+1} - Erro Médio: {erro_medio:.2f} anos de diferença.")

    print("\nTreinamento Concluído!")

    # salva os pesos
    torch.save(modelo.state_dict(), '/content/drive/MyDrive/pesos_tcc_openbhb.pth')
    print("Pesos salvos com sucesso no seu Google Drive!")

if __name__ == "__main__":
    treinar_modelo()

In [ ]:
import torch
import numpy as np

def laudo(imagem, pesos, idade_real):
    print("Iniciando carregamento do modelo...")

    modelo = CNN3D_OpenBHB()
    modelo.load_state_dict(torch.load(pesos, map_location=torch.device('cpu')))
    modelo.eval()

    volume_mri = np.load(imagem)
    tensor_mri = torch.from_numpy(volume_mri).float()

    tensor_mri = tensor_mri.view(1, 1, 121, 145, 121)

    print("Gerando predição (Isso pode levar alguns segundos na CPU)...")
    with torch.no_grad():
        predicao = modelo(tensor_mri)
        idade_predita = predicao.item()

    bag = idade_predita - idade_real

    print("-" * 50)
    print("      RESULTADO DA PREDIÇÃO - BAG      ")
    print("-" * 50)
    print(f"Idade Cronológica (Real): {idade_real:.2f} anos")
    print(f"Idade Biológica Predita:  {idade_predita:.2f} anos")

    if bag > 0:
        print(f"Brain Age Gap (BAG):      +{bag:.2f} anos (Acelerado)")
    else:
        print(f"Brain Age Gap (BAG):      {bag:.2f} anos (Preservado)")
    print("-" * 50)


PESOS_TREINADOS = "/content/drive/MyDrive/pesos_tcc_openbhb.pth"
IMAGEM_PACIENTE = "/content/drive/MyDrive/Colab_Notebooks/dataset/sub-277976850330_preproc-cat12vbm_desc-gm_T1w.npy"
IDADE_CRONOLOGICA = 78.35728952772074

laudo(IMAGEM_PACIENTE, PESOS_TREINADOS, IDADE_CRONOLOGICA)